# 03 — Event Cleaning & Core Dimensions

Phase 2 pipeline: raw JSON → clean parquet files with QA validation.

**Outputs:**
- `dim_product_code.parquet` — device classification dimension
- `clean_event_device_level.parquet` — adverse events at device-event grain
- `dim_manufacturer_alias.parquet` — manufacturer name standardization

In [ ]:
import sys

sys.path.insert(0, "..")

from src.cleaning.adverse_events import clean_adverse_events
from src.cleaning.classification import build_dim_product_code
from src.mapping.manufacturer import build_manufacturer_alias
from src.qa.checks import (
    check_coverage,
    check_null_rate,
    check_row_count,
    check_uniqueness,
    check_volume_shift,
    run_checks,
)

## 1. Build Classification Dimension

In [ ]:
dim_pc = build_dim_product_code()
print(f"Shape: {dim_pc.shape}")
print(f"Device classes: {dim_pc['device_class'].value_counts().to_dict()}")
dim_pc.head()

## 2. Clean Adverse Events

In [ ]:
events = clean_adverse_events()
print(f"Clean event rows: {len(events):,}")
print(f"Unique mdr_report_keys: {events['mdr_report_key'].nunique():,}")
print(f"Unique event_record_ids: {events['event_record_id'].nunique():,}")
events.head()

## 3. Dedup Analysis

In [ ]:
total = len(events)
latest_only = events[events["is_latest_version"]].copy()
deduped = len(latest_only)

print(f"Total device-event rows: {total:,}")
print(f"Latest-version only: {deduped:,}")
print(f"Followup/superseded: {total - deduped:,} ({(total - deduped) / total:.1%})")

# Year distribution
print("\nRows by year (latest version only):")
print(latest_only.groupby("event_year").size().to_string())

## 4. Build Manufacturer Alias

In [ ]:
alias = build_manufacturer_alias()
raw_count = alias["raw_name"].nunique()
std_count = alias["standardized_name"].nunique()
print(f"Raw manufacturer names: {raw_count:,}")
print(f"Standardized names: {std_count:,}")
print(f"Reduction: {(1 - std_count / raw_count):.1%}" if raw_count > 0 else "No names")
print("\nRule distribution:")
print(alias["normalization_rule"].value_counts().to_string())
print(f"\nManual review flagged: {alias['manual_review_flag'].sum():,}")
alias.head(10)

## 5. QA Summary

In [ ]:
checks = [
    # Classification dimension
    check_row_count(dim_pc, "dim_product_code", min_rows=100),
    check_uniqueness(dim_pc, ["product_code"], "dim_product_code_pk"),
    check_null_rate(dim_pc, "device_name"),
    check_null_rate(dim_pc, "device_class"),
    # Adverse events
    check_row_count(latest_only, "clean_events_latest", min_rows=1000),
    check_uniqueness(events, ["event_record_id"], "event_record_id_pk"),
    check_null_rate(latest_only, "product_code", max_rate=0.30),
    check_null_rate(latest_only, "date_received", max_rate=0.01),
    check_null_rate(latest_only, "manufacturer_d_name", max_rate=0.10),
    check_volume_shift(latest_only, "event_year"),
    # Manufacturer alias
    check_row_count(alias, "manufacturer_alias", min_rows=10),
    # Coverage: product codes in events vs classification dim
    check_coverage(
        latest_only,
        "product_code",
        reference_values=set(dim_pc["product_code"].dropna()),
    ),
]

summary = run_checks(checks)
summary.style.map(
    lambda v: (
        "background-color: #d4edda"
        if v == "pass"
        else ("background-color: #fff3cd" if v == "warn" else ("background-color: #f8d7da" if v == "fail" else ""))
    ),
    subset=["status"],
)

## 6. Data Profile

In [ ]:
# Grain verification
print("=== Grain Verification ===")
print(f"event_record_id unique: {events['event_record_id'].is_unique}")
print(f"dim_product_code PK unique: {dim_pc['product_code'].is_unique}")

# Product code coverage
pc_in_events = set(latest_only["product_code"].dropna().unique())
pc_in_dim = set(dim_pc["product_code"].dropna().unique())
matched = len(pc_in_events & pc_in_dim)
print("\n=== Product Code Coverage ===")
print(f"Unique codes in events: {len(pc_in_events):,}")
print(f"Codes in classification dim: {len(pc_in_dim):,}")
print(f"Matched: {matched:,} ({matched / len(pc_in_events):.1%} of event codes)" if pc_in_events else "No codes")

# Manufacturer coverage
raw_mfg = latest_only["manufacturer_d_name"].dropna().nunique()
if len(alias) > 0:
    aliased = alias["raw_name"].nunique()
    print("\n=== Manufacturer Coverage ===")
    print(f"Unique raw names in events: {raw_mfg:,}")
    print(f"Names in alias table: {aliased:,}")
    print(f"Coverage: {aliased / raw_mfg:.1%}" if raw_mfg > 0 else "N/A")

# Seriousness distribution
print("\n=== Seriousness (latest version) ===")
print(f"Deaths: {latest_only['has_death'].sum():,} ({latest_only['has_death'].mean():.1%})")
print(f"Serious injuries: {latest_only['has_serious_injury'].sum():,} ({latest_only['has_serious_injury'].mean():.1%})")
print(f"Malfunctions: {latest_only['has_malfunction'].sum():,} ({latest_only['has_malfunction'].mean():.1%})")

# Year distribution
print("\n=== Year Distribution ===")
print(latest_only["event_year"].value_counts().sort_index().to_string())